# 04. Chain of Density (CoD) 요약

- 논문: https://arxiv.org/pdf/2309.04269

1. 먼저 엔티티(핵심 개체)가 적은 **짧은 요약**을 만듭니다.
2. 요약에서 **빠진 중요 엔티티**(missing entities)를 찾아,
3. **길이는 유지한 채** 엔티티를 추가해 더 "밀도 높은" 요약으로 다시 씁니다.
4. 2~3 을 `iterations` 번 반복합니다.

→ 길이는 같지만 정보량이 점점 많아지는 요약들을 얻습니다. 연구 결과 사람들은 **중간 정도 밀도**의 요약을 가장 선호했습니다.

## 환경 설정

- API 키(`OPENAI_API_KEY`, `LANGSMITH_API_KEY` 등)는 `.env` 파일 또는 시스템 환경변수에서 읽어옵니다.
- `logging.langsmith()` 를 호출하면 실행 과정이 LangSmith 에 기록되어 체인 내부 동작을 확인할 수 있습니다.

In [1]:
# .env 파일에 저장된 API KEY 를 환경변수로 불러옵니다.
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (프로젝트 이름: Summary)
from langchain_teddynote import logging

logging.langsmith("Summary")

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary


## 데이터 로드

**실습 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호 (`data/SPRI_AI_Brief_2023년12월호_F.pdf`)

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 를 페이지 단위로 로드합니다. (1페이지 = Document 1개)
loader = PyPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
print(f"전체 페이지 수: {len(docs)}")

# 실습 비용/시간을 줄이기 위해 일부 페이지(4~8쪽)만 요약합니다.
docs = docs[3:8]
print(f"요약할 페이지 수: {len(docs)}")

C:\Users\user\AppData\Local\Temp\ipykernel_2040\1564795920.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


전체 페이지 수: 23
요약할 페이지 수: 5


### 프롬프트 가져오기 (LangSmith Hub)

> ⚠️ **레거시 변경점**
> 예전 코드의 `from langchain import hub` / `hub.pull(...)` 은 LangChain 1.x 에서 `langchain_classic` 으로 밀려났고,
> 최신 `langsmith` 는 **다른 사람(owner)의 공개 프롬프트**를 기본적으로 막습니다.
> 그래서 `langsmith.Client().pull_prompt(..., dangerously_pull_public_prompt=True)` 로 직접 가져옵니다.
> (`teddynote` 튜토리얼 프롬프트처럼 **신뢰할 수 있는 프롬프트에만** 사용하세요.)

In [3]:
from langsmith import Client

client = Client()


def pull_prompt(name: str):
    """LangSmith Hub 에서 공개 프롬프트를 가져옵니다. (hub.pull 대체)"""
    return client.pull_prompt(name, dangerously_pull_public_prompt=True)

## CoD 프롬프트

**입력 파라미터**

| 파라미터 | 설명 | 기본값 |
|---|---|---|
| `content` | 요약할 콘텐츠 | (필수) |
| `content_category` | 콘텐츠 종류 (기사, 녹취록, 블로그, 논문 등) | `Article` |
| `entity_range` | 한 번에 추가할 엔티티 수 범위 | `1-3` |
| `max_words` | 요약 1개의 최대 단어 수 | `80` |
| `iterations` | 밀도 높이기 반복 횟수 (총 요약 수 = 반복 + 1) | `5` |

LLM 은 `[{"missing_entities": "...", "denser_summary": "..."}, ...]` 형태의 **JSON 리스트**로 답합니다.

In [4]:
cod_prompt = pull_prompt("teddynote/chain-of-density-prompt")
cod_prompt.pretty_print()

================================ System Message ================================

As an expert copy-writer, you will write increasingly concise, entity-dense summaries of the user provided {content_category}. The initial summary should be under {max_words} words and contain {entity_range} informative Descriptive Entities from the {content_category}.

A Descriptive Entity is:
- Relevant: to the main story.
- Specific: descriptive yet concise (5 words or fewer).
- Faithful: present in the {content_category}.
- Anywhere: located anywhere in the {content_category}.

# Your Summarization Process
- Read through the {content_category} and the all the below sections to get an understanding of the task.
- Pick {entity_range} informative Descriptive Entities from the {content_category} (";" delimited, do not add spaces).
- In your output JSON list of dictionaries, write an initial summary of max {max_words} words containing the Entities.
- You now have `[{"missing_entities": "...", "denser_summa

In [5]:
from langchain_core.output_parsers import SimpleJsonOutputParser
from langchain_openai import ChatOpenAI

# content 를 제외한 입력값은 없으면 기본값을 채워 넣습니다.
cod_chain_inputs = {
    "content": lambda d: d.get("content"),
    "content_category": lambda d: d.get("content_category", "Article"),
    "entity_range": lambda d: d.get("entity_range", "1-3"),
    "max_words": lambda d: int(d.get("max_words", 80)),
    "iterations": lambda d: int(d.get("iterations", 5)),
}

# 입력 기본값 채우기 → 프롬프트 → LLM → JSON 파싱
cod_chain = (
    cod_chain_inputs
    | cod_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | SimpleJsonOutputParser()
)

# 최종(가장 밀도 높은) 요약만 꺼내는 체인
cod_final_summary_chain = cod_chain | (
    lambda output: output[-1].get(
        "denser_summary", '오류: 마지막 딕셔너리에 "denser_summary" 키가 없습니다'
    )
)

## 요약할 데이터 확인

In [6]:
content = docs[1].page_content
print(content)

SPRi AI Brief |  
2023-12월호
2
G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의
n G7이 첨단 AI 시스템을 개발하는 기업을 대상으로 AI 위험 식별과 완화를 위해 자발적인 
채택을 권고하는 AI 국제 행동강령을 마련
n 행동강령은 AI 수명주기 전반에 걸친 위험 평가와 완화, 투명성과 책임성의 보장, 정보공유와 
이해관계자 간 협력, 보안 통제, 콘텐츠 인증과 출처 확인 등의 조치를 요구
KEY Contents
£ G7, 첨단 AI 시스템의 위험 관리를 위한 국제 행동강령 마련
n 주요 7개국(G7)*은 2023년 10월 30일 ‘히로시마 AI 프로세스’를 통해 AI 기업 대상의 AI 국제 
행동강령(International Code of Conduct for Advanced AI Systems)에 합의
∙ G7은 2023년 5월 일본 히로시마에서 개최된 정상회의에서 생성 AI에 관한 국제규범 마련과 
정보공유를 위해 ‘히로시마 AI 프로세스’를 출범**
∙ 기업의 자발적 채택을 위해 마련된 이번 행동강령은 기반모델과 생성 AI를 포함한 첨단 AI 시스템의 
위험 식별과 완화에 필요한 조치를 포함
* 주요 7개국(G7)은 미국, 일본, 독일, 영국, 프랑스, 이탈리아, 캐나다를 의미
** 5월 정상회의에는 한국, 호주, 베트남 등을 포함한 8개국이 초청을 받았으나, AI 국제 행동강령에는 우선 G7 국가만 포함하여 채택
n G7은 행동강령을 통해 아래의 조치를 제시했으며, 빠르게 발전하는 기술에 대응할 수 있도록 
이해관계자 협의를 통해 필요에 따라 개정할 예정
∙ 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택하고, 
첨단 AI 시스템의 출시와 배포 이후 취약점과 오용 사고, 오용 유형을 파악해 완화
∙ 첨단 AI 시스템의 성능과 한계를 공개하고 적절하거나 부적절한 사용영역을 알리는 방법으로 투명성을 
보장하고 책임성을 강화
∙ 산업계, 정부, 시민사회, 학

## 스트리밍 실행

`SimpleJsonOutputParser` 는 스트리밍 중에도 **지금까지 완성된 JSON** 을 계속 돌려줍니다.
각 청크는 이전 청크에 내용이 덧붙은 전체 리스트이므로, `\r`(캐리지 리턴)로 같은 줄을 덮어쓰며 출력합니다.

In [7]:
import textwrap

results: list[dict[str, str]] = []

# 스트리밍: 매번 "지금까지 파싱된 전체 리스트"가 들어오므로 마지막 값만 유지하면 됩니다.
for partial_json in cod_chain.stream({"content": content, "content_category": "Article"}):
    results = partial_json
    print(results, end="\r", flush=True)

total_summaries = len(results)
print("\n")

# 단계별 요약 출력
for i, cod in enumerate(results, start=1):
    # 이번 단계에서 추가된 엔티티 (";" 로 구분됨)
    added_entities = ", ".join(
        ent.strip()
        for ent in cod.get("missing_entities", 'ERR: "missing_entities" key not found').split(";")
    )
    summary = cod.get("denser_summary", 'ERR: missing key "denser_summary"')

    print(f"### CoD Summary {i}/{total_summaries}, 추가된 엔티티(entity): {added_entities}\n")
    print(textwrap.fill(summary, width=80) + "\n")

# 반복문이 끝난 뒤 마지막 summary 가 최종 요약입니다.
print("\n============== [최종 요약] =================\n")
print(summary)

[{'missing_entities': 'G7;AI 국제 행동강령;히로시마 AI 프로세스', 'denser_summary': 'G7은 2023년 10월 30일 히로시마 AI 프로세스를 통해 AI 기업을 위한 AI 국제 행동강령을 마련했다. 이 행동강령은 AI 위험 식별과 완화를 위한 자발적 채택을 권장하며, AI 수명주기 전반의 위험 평가, 투명성, 책임성, 정보공유, 보안 통제, 콘텐츠 인증 등의 조치를 요구한다.'}, {'missing_entities': '위험 평가;투명성;책임성', 'denser_summary': 'G7은 2023년 10월 30일 히로시마 AI 프로세스를 통해 AI 기업을 위한 AI 국제 행동강령을 마련했다. 이 행동강령은 AI 위험 식별과 완화를 위한 자발적 채택을 권장하며, AI 수명주기 전반의 위험 평가, 투명성, 책임성, 정보공유, 보안 통제, 콘텐츠 인증 등의 조치를 요구한다.'}, {'missing_entities': '정보공유;보안 통제;콘텐츠 인증', 'denser_summary': 'G7은 2023년 10월 30일 히로시마 AI 프로세스를 통해 AI 기업을 위한 AI 국제 행동강령을 마련했다. 이 행동강령은 AI 위험 식별과 완화를 위한 자발적 채택을 권장하며, AI 수명주기 전반의 위험 평가, 투명성, 책임성, 정보공유, 보안 통제, 콘텐츠 인증 등의 조치를 요구한다.'}, {'missing_entities': 'AI 수명주기;위험 관리 정책;강력한 보안', 'denser_summary': 'G7은 2023년 10월 30일 히로시마 AI 프로세스를 통해 AI 기업을 위한 AI 국제 행동강령을 마련했다. 이 행동강령은 AI 위험 식별과 완화를 위한 자발적 채택을 권장하며, AI 수명주기 전반의 위험 평가, 투명성, 책임성, 정보공유, 강력한 보안, 콘텐츠 인증 등의 조치를 요구한다.'}, {'missing_entities': 'AI 거버넌스;사회적 위험;기후 위기', 'denser_summary': 'G7은 2023년 1

최종 요약만 필요하다면 `cod_final_summary_chain` 을 사용합니다.

In [8]:
print(cod_final_summary_chain.invoke({"content": content}))

G7은 2023년 10월 30일 히로시마 AI 프로세스를 통해 AI 기업을 위한 AI 국제 행동강령을 마련했다. 이 행동강령은 AI 위험 식별과 완화를 위한 자발적 채택을 권장하며, AI 수명주기 전반의 위험 평가, 투명성, 정보공유, 보안 통제, 콘텐츠 인증, 위험 완화, AI 거버넌스, 사이버보안, 사회적 위험, 기후 위기, 세계적 난제 해결을 위한 조치를 요구한다.
